In [16]:
# ! pip install paramiko

In [17]:
# download_annotations.py
import os
import stat
import time
import paramiko

from dotenv import load_dotenv
from getpass import getpass
from pathlib import Path


In [18]:
load_dotenv() 

True

In [19]:
# ====== CONFIG ======
HOST = os.getenv("HOST")
PORT = os.getenv("PORT")
USERNAME = os.getenv("USERNAME")

KEY_FILE = os.path.expanduser(os.getenv("PASSKEY_PATH"))  # ou None para usar só agent/senha
PASSWORD = None  # só use se não houver chave (menos seguro)

TODAY = time.strftime("%Y%m%d")

REMOTE_DIR = os.getenv("REMOTE_DIR")
LOCAL_DIR = Path(f"data/annotations/{TODAY}")
# ====================

def ensure_local_dir(path: Path):
    path.mkdir(parents=True, exist_ok=True)

def files_equal_by_meta(local: Path, sftp, remote_path: str):
    try:
        rstat = sftp.stat(remote_path)
    except FileNotFoundError:
        return False
    if not local.exists():
        return False
    lst = local.stat()
    same_size = lst.st_size == rstat.st_size
    same_mtime = int(lst.st_mtime) == int(rstat.st_mtime)
    return same_size and same_mtime

def sftp_walk(sftp, remote_path):
    for entry in sftp.listdir_attr(remote_path):
        mode = entry.st_mode
        name = entry.filename
        if stat.S_ISDIR(mode):
            yield from sftp_walk(sftp, remote_path.rstrip("/") + "/" + name)
        else:
            yield remote_path, name

def _load_pkey(path: str):
    """
    Tenta carregar a chave privada (Ed25519 ou RSA).
    Se precisar de passphrase e não foi fornecida, pergunta no terminal.
    """
    if not path:
        return None
    # tenta Ed25519
    try:
        return paramiko.Ed25519Key.from_private_key_file(path)
    except paramiko.ssh_exception.PasswordRequiredException:
        pw = getpass(f"Passphrase para {path}: ")
        return paramiko.Ed25519Key.from_private_key_file(path, password=pw)
    except Exception:
        pass
    # tenta RSA
    try:
        return paramiko.RSAKey.from_private_key_file(path)
    except paramiko.ssh_exception.PasswordRequiredException:
        pw = getpass(f"Passphrase para {path}: ")
        return paramiko.RSAKey.from_private_key_file(path, password=pw)

def main():
    ensure_local_dir(LOCAL_DIR)

    client = paramiko.SSHClient()
    client.set_missing_host_key_policy(paramiko.AutoAddPolicy())

    # 1) tenta agente primeiro (chaves já adicionadas via ssh-agent)
    agent = paramiko.Agent()
    agent_keys = agent.get_keys()

    pkey = _load_pkey(KEY_FILE) if KEY_FILE else None

    connect_kwargs = dict(
        hostname=HOST,
        port=PORT,
        username=USERNAME,
        timeout=20,
        look_for_keys=False,   # vamos controlar explicitamente
        allow_agent=True,      # permite usar agent se disponível
    )

    if pkey:
        connect_kwargs["pkey"] = pkey
    elif agent_keys:
        # se não há KEY_FILE mas há chaves no agente, Paramiko tentará usá-las
        pass
    elif PASSWORD:
        connect_kwargs["password"] = PASSWORD
    else:
        raise RuntimeError("Nenhuma forma de autenticação configurada (KEY_FILE, agent ou PASSWORD).")

    print(f"Conectando ao servidor ...")
    client.connect(**connect_kwargs)
    sftp = client.open_sftp()

    total_downloaded, total_skipped = 0, 0
    print(f"Baixando para {LOCAL_DIR.resolve()}")

    # percorre recursivamente
    for rdir, fname in sftp_walk(sftp, REMOTE_DIR):
        rel = os.path.relpath(rdir, REMOTE_DIR)
        ldir = LOCAL_DIR / ("" if rel == "." else rel)
        ensure_local_dir(ldir)

        rpath = rdir.rstrip("/") + "/" + fname
        lpath = ldir / fname

        if files_equal_by_meta(lpath, sftp, rpath):
            total_skipped += 1
            continue

        print(f"- baixando: {lpath}")
        sftp.get(rpath, str(lpath))
        rst = sftp.stat(rpath)
        os.utime(lpath, (int(time.time()), int(rst.st_mtime)))
        total_downloaded += 1

    sftp.close()
    client.close()
    print(f"Concluído. Baixados: {total_downloaded}, pulados: {total_skipped}")

